# 01 · EDA and Feature Sanity

**Purpose:** before running any ablation, verify three things on the 822K-row research panel

1. **Data quality** — timestamps, missingness, outliers
2. **Feature distributions** — `ret_1m`, `rolling_vol`, `vwap_dev`, `volume_surprise`, `trend_strength`
3. **Forward-return IC** — do these features actually have any predictive power for `fwd_ret_{5,15,30}m`?

If every feature IC is ~0, the panel itself is broken and no amount of ablation will save it.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

PANEL_PATH = PROJECT / 'data_qtc' / 'combined' / 'research_panel.parquet'
print('Loading', PANEL_PATH)
panel = pd.read_parquet(PANEL_PATH)
print(f'shape: {panel.shape}')
print(f'symbols: {sorted(panel["symbol"].unique().tolist())}')
print(f'date range: {panel["timestamp"].min()}  ->  {panel["timestamp"].max()}')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

## 1. Shape per symbol and date coverage

In [ ]:
summary = (
    panel.groupby('symbol')
    .agg(
        n_bars=('timestamp', 'size'),
        start=('timestamp', 'min'),
        end=('timestamp', 'max'),
    )
    .sort_values('n_bars', ascending=False)
)
summary

In [ ]:
missing = (
    panel.isna().mean().mul(100).round(2)
    .sort_values(ascending=False)
    .head(25)
    .to_frame('pct_missing')
)
missing.style.background_gradient(cmap='Reds')

Most NaNs are the expected warm-up rows for the rolling windows (`ma_slow(60)` needs 60 bars, `rolling_vol(20)` needs 20, etc.) and for `fwd_ret_*` on the last rows of each symbol (no future to peek at).

## 2. Price trajectories (downsampled)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=False)
axes = axes.ravel()
for ax, sym in zip(axes, sorted(panel['symbol'].unique())):
    sub = panel.loc[panel['symbol'] == sym, ['timestamp', 'close']].copy()
    sub = sub.set_index('timestamp').resample('1h').last().dropna()
    ax.plot(sub.index, sub['close'].values, lw=0.8)
    ax.set_title(f'{sym} (hourly resample)')
    ax.tick_params(axis='x', rotation=30)
for ax in axes[len(panel['symbol'].unique()):]:
    ax.set_visible(False)
fig.tight_layout()

## 3. Feature distributions (NVDA only, clipped to avoid tails eating the plot)

In [ ]:
nvda = panel[panel['symbol'] == 'NVDA'].copy()
feat_cols = ['ret_1m', 'rolling_vol', 'vwap_dev', 'volume_surprise', 'trend_strength']

fig, axes = plt.subplots(1, len(feat_cols), figsize=(18, 3))
for ax, c in zip(axes, feat_cols):
    x = nvda[c].dropna()
    lo, hi = np.quantile(x, [0.01, 0.99])
    sns.histplot(x.clip(lo, hi), bins=60, ax=ax, kde=False, color='steelblue')
    ax.set_title(c)
fig.tight_layout()

## 4. Forward-return IC

Spearman rank IC between each feature and `fwd_ret_15m`, per symbol. A usable alpha typically has IC in the 0.01–0.05 range; anything above 0.05 on a noisy 1-minute horizon is suspicious.

In [ ]:
feature_candidates = [
    'trend_strength', 'fast_minus_slow', 'price_minus_fast',
    'vwap_dev', 'volume_surprise', 'abs_ret_5m', 'hl_range',
    'SPY_trend_strength', 'QQQ_trend_strength', 'XLK_trend_strength',
]
horizons = ['fwd_ret_5m', 'fwd_ret_15m', 'fwd_ret_30m']

rows = []
for sym, g in panel.groupby('symbol'):
    g = g.replace([np.inf, -np.inf], np.nan).dropna(subset=horizons)
    for f in feature_candidates:
        if f not in g.columns:
            continue
        x = g[f].values
        ok = np.isfinite(x)
        for h in horizons:
            y = g[h].values
            mask = ok & np.isfinite(y)
            if mask.sum() < 1000:
                rows.append({'symbol': sym, 'feature': f, 'horizon': h, 'IC': np.nan, 'n': int(mask.sum())})
                continue
            rho, _ = spearmanr(x[mask], y[mask])
            rows.append({'symbol': sym, 'feature': f, 'horizon': h, 'IC': rho, 'n': int(mask.sum())})

ic = pd.DataFrame(rows)
ic_wide = (
    ic.pivot_table(index=['feature', 'symbol'], columns='horizon', values='IC')
    [horizons]
    .sort_index()
)
ic_wide.style.background_gradient(cmap='RdBu_r', vmin=-0.05, vmax=0.05).format('{:.4f}')

In [ ]:
ic_by_feature = (
    ic.groupby(['feature', 'horizon'])['IC'].mean().unstack('horizon')[horizons]
    .sort_values('fwd_ret_15m', key=lambda s: s.abs(), ascending=False)
)
print('Cross-symbol mean IC (top = strongest signal):')
ic_by_feature

## 5. Session-time effect

Do returns / vol / signal quality differ by minute of the NY session? This motivates `is_open_window / is_midday / is_late_day` as gating features.

In [ ]:
hod = (
    nvda.assign(abs_ret=nvda['ret_1m'].abs())
    .groupby('minute_of_day_ny')
    .agg(mean_ret=('ret_1m', 'mean'), mean_abs_ret=('abs_ret', 'mean'),
         vol=('rolling_vol', 'mean'), volume=('volume', 'mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, col, title in zip(
    axes,
    ['mean_abs_ret', 'vol', 'volume'],
    ['mean |ret_1m|', 'mean rolling_vol', 'mean volume'],
):
    ax.plot(hod['minute_of_day_ny'], hod[col], lw=1)
    ax.set_title(title)
    ax.set_xlabel('minute of NY day')
fig.tight_layout()

## 6. Autocorrelation of returns and squared returns

- Mean returns should be near zero at all lags (efficient market).
- Squared returns show strong positive autocorrelation (volatility clustering) — this is the empirical basis for vol targeting and dynamic stops.

In [ ]:
from pandas.plotting import autocorrelation_plot

def acf(x, n_lags=40):
    x = np.asarray(x)
    x = x - x.mean()
    var = (x ** 2).mean()
    return np.array([((x[:-k] * x[k:]).mean() / var) if k > 0 else 1.0 for k in range(n_lags + 1)])

ret = nvda['ret_1m'].dropna().values
lags = np.arange(41)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(lags, acf(ret, 40), color='steelblue')
axes[0].set_title('ACF of ret_1m (should hover near 0)')
axes[0].set_xlabel('lag (minutes)')
axes[1].bar(lags, acf(ret**2, 40), color='indianred')
axes[1].set_title('ACF of squared ret_1m (volatility clustering)')
axes[1].set_xlabel('lag (minutes)')
fig.tight_layout()

## What to carry into `02_ablations.ipynb`

1. **NVDA has clean minute data for the full window** — we use it as the single-asset target per the `B0` spec.
2. **`trend_strength` / `fast_minus_slow`** are the strongest individual features by IC on 15m forward returns.
3. **Volume-surprise and VWAP-deviation** show weak but non-zero IC — useful as confirmation features, not as primary drivers.
4. **Open window and late-day bars** are structurally different (higher absolute returns and volatility) — this supports session-based gating in the execution layer.
5. **Squared-return ACF decays slowly**, confirming that vol targeting and vol-aware stops are well-motivated on this data.